## MODELO REGRESIÓN LOGÍSTICA ANÁLISIS DE ASPECTOS

CARGAR DATOS

In [1]:
import pandas as pd
df = pd.read_excel(r"Z:\00 COMUN PRESTACIONES Y PROVEEDORES\NPS_POWER_BI\Copias\00_NPS\Pruebas\analisis_salud.xlsx")

In [2]:
df = df[["Surveyid for internal use (e.g. RI link)", "texto_norm", "aspectos_absa"]].copy()

import ast
df["aspectos_absa"] = df["aspectos_absa"].apply(ast.literal_eval)

df = df[df["aspectos_absa"].apply(len) > 0]

df = df[
    df["texto_norm"].notna() &
    (df["texto_norm"].str.strip() != "")
]

In [6]:
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
stop_words = set(stopwords.words("spanish"))

def limpiar_a_texto(texto):
    if not isinstance(texto, str):
        return ""

    texto = texto.lower()
    texto = texto.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(texto)
    tokens = [
        t for t in tokens
        if t.isalpha() and t not in stop_words and len(t) > 2
    ]
    return " ".join(tokens)

df["texto_modelo"] = df["texto_norm"].apply(limpiar_a_texto)

df = df[df["texto_modelo"].str.strip() != ""]

In [7]:
X = df["texto_modelo"]
y = df["aspectos_absa"]

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y_bin = mlb.fit_transform(y)
print("Aspectos detectados:")
print(mlb.classes_)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_bin,
    test_size=0.2,
    random_state=42
)

from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9,
    sublinear_tf=True
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    )
)
model.fit(X_train_tfidf, y_train)

from sklearn.metrics import classification_report
y_pred = model.predict(X_test_tfidf)
print(
    classification_report(
        y_test,
        y_pred,
        target_names=mlb.classes_
    )
)

Aspectos detectados:
['Comunicación' 'Precio' 'Producto' 'Servicio' 'Sistema' 'Velocidad']
              precision    recall  f1-score   support

Comunicación       0.97      0.95      0.96      2307
      Precio       0.94      0.94      0.94      1989
    Producto       0.99      0.96      0.98      4304
    Servicio       1.00      0.96      0.98     10775
     Sistema       0.95      0.96      0.95      1127
   Velocidad       1.00      0.95      0.98      8312

   micro avg       0.99      0.96      0.97     28814
   macro avg       0.97      0.95      0.96     28814
weighted avg       0.99      0.96      0.97     28814
 samples avg       0.99      0.97      0.97     28814



C:\Users\ACOYLO1\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.9,
        sublinear_tf=True
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced"
        )
    ))
])


from sklearn.model_selection import cross_val_score

scores = cross_val_score(
    pipeline,
    X,
    y_bin,
    cv=5,
    scoring="f1_macro"
)

print("F1 macro por fold:", scores)
print("F1 macro medio:", scores.mean())
print("Desviación:", scores.std())

F1 macro por fold: [0.96529314 0.96306858 0.95950608 0.96735589 0.96561558]
F1 macro medio: 0.9641678549798727
Desviación: 0.0027006034401619405


In [9]:
import joblib

joblib.dump(model, "modelo_aspectos.pkl")
joblib.dump(vectorizer, "tfidf_aspectos.pkl")
joblib.dump(mlb, "mlb_aspectos.pkl")

['mlb_aspectos.pkl']

In [ ]:
import joblib

model = joblib.load("modelo_aspectos.pkl")
vectorizer = joblib.load("tfidf_aspectos.pkl")
mlb = joblib.load("mlb_aspectos.pkl")